In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import loguniform, randint, uniform
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    validation_curve,
    GridSearchCV,
    RandomizedSearchCV,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    recall_score,
    precision_score,
    f1_score,
    silhouette_score,
)
from xgboost import XGBClassifier
import joblib

In [ ]:
import os

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))    

In [ ]:
df = pd.read_csv("/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print(df.shape)
print(df["TotalCharges"].isna().sum())
df.head()

In [ ]:
path = "/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(path)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0)  # blanks have tenure = 0


def build_features(data):
    X = data.drop(columns=["customerID", "Churn"])
    return pd.get_dummies(X, drop_first=True).astype(float)


y = (df["Churn"] == "Yes").astype(int)
X = build_features(df)

# The test set is locked away until Part 7. Do not touch it before then.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

In [ ]:
lr = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000)),
])

scores = []
for seed in range(20):
    Xa, Xb, ya, yb = train_test_split(
        X_train, y_train, test_size=0.25, random_state=seed, stratify=y_train
    )
    scores.append(accuracy_score(yb, lr.fit(Xa, ya).predict(Xb)))
scores = np.array(scores)

print(f"min {scores.min():.3f}  max {scores.max():.3f}  std {scores.std():.4f}")

p, n = scores.mean(), len(yb)
se = np.sqrt(p * (1 - p) / n)
print(f"Theoretical standard error: {se:.4f}  -> 95% CI +/- {1.96 * se:.3f}")

plt.bar(range(20), scores)
plt.ylim(scores.min() - 0.02, scores.max() + 0.02)
plt.xlabel("random_state")
plt.ylabel("Validation accuracy")
plt.title("Same model, same data, different split")
plt.show()

**Insight (Task 1.3):** Across 20 random splits, the same Logistic Regression scored between 0.780 (min) and 0.828 (max) accuracy, with a standard deviation of 0.0104. The theoretical standard error is 0.0107, giving a 95% interval of about ±0.021. The observed spread almost exactly matches the theory, so it is sampling noise from the small validation set.

Yes, two students could reach opposite conclusions. The best and worst seeds differ by 0.048, more than double the ±0.021 interval, so a student who got a lucky split for LR and an unlucky one for RF could conclude LR is better, while another student with the opposite luck would conclude the reverse. A single split cannot separate a real model difference from noise, so we use 5-fold CV from here on.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
metrics = ["roc_auc", "recall", "f1"]


def cv_report(name, model):
    r = cross_validate(model, X_train, y_train, cv=cv, scoring=metrics)
    row = {"model": name}
    for m in metrics:
        s = r["test_" + m]
        row[m] = f"{s.mean():.3f} +/- {s.std():.3f}"
    return row


rf = RandomForestClassifier(
    n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=42
)

print(
    pd.DataFrame(
        [cv_report("Logistic Regression", lr), cv_report("Random Forest", rf)]
    ).to_string(index=False)
)

**Insight (Part 2):**

| model | roc_auc | recall | f1 |
|---|---|---|---|
| Logistic Regression | 0.846 ± 0.013 | 0.545 ± 0.042 | 0.594 ± 0.030 |
| Random Forest | 0.844 ± 0.011 | 0.496 ± 0.019 | 0.573 ± 0.020 |

(all values 5-fold CV mean ± std)

**Is the difference larger than the standard deviations?** For AUC, no. The gap is 0.002 against stds of 0.013 and 0.011, so LR and RF are indistinguishable. For F1 the gap is 0.021, which is smaller than LR's std (0.030) and about equal to RF's (0.020), so it is not a clear difference either. Recall has the largest gap, 0.049, which is just above LR's std (0.042) and above RF's (0.019). LR looks slightly better on recall, but with only 5 folds I would not call it a decisive win.

**What recall tells us that AUC does not:** AUC measures how well a model ranks churners above non-churners across all thresholds, so both models look strong (about 0.85). Recall measures what happens at the actual decision threshold (0.5 by default): both models catch only about half of the real churners (0.545 and 0.496), so roughly 45–50% of customers who will leave are missed. Since a missed churner is the costly mistake, a good AUC hides a weak operating point that recall exposes.

In [ ]:
Cs = np.logspace(-4, 2, 13)
tr, va = validation_curve(
    lr, X_train, y_train, param_name="model__C", param_range=Cs, cv=cv, scoring="roc_auc"
)

plt.semilogx(Cs, tr.mean(axis=1), "o-", label="Train AUC")
plt.semilogx(Cs, va.mean(axis=1), "o-", label="CV AUC")
plt.xlabel("C  (smaller = stronger regularization)")
plt.ylabel("AUC")
plt.legend()
plt.title("Validation curve: Logistic Regression")
plt.show()

print("Best C by CV:", Cs[va.mean(axis=1).argmax()])

In [ ]:
grid = {
    "max_depth": [4, 8, 12, None],
    "min_samples_leaf": [1, 5, 20],
    "max_features": ["sqrt", 0.5],
}
print("Combinations:", 4 * 3 * 2, " Fits:", 4 * 3 * 2 * 5)

t0 = time.time()
gs = GridSearchCV(
    RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
    grid,
    cv=cv,
    scoring="roc_auc",
)
gs.fit(X_train, y_train)

print(f"Grid:   best AUC {gs.best_score_:.4f}  in {time.time() - t0:.0f}s")
print(gs.best_params_)

In [ ]:
dist = {
    "max_depth": randint(3, 20),
    "min_samples_leaf": randint(1, 40),
    "max_features": uniform(0.1, 0.8),  # fraction of features
}

t0 = time.time()
rs = RandomizedSearchCV(
    RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
    dist,
    n_iter=24,
    cv=cv,
    scoring="roc_auc",
    random_state=42,
)
rs.fit(X_train, y_train)

print(f"Random: best AUC {rs.best_score_:.4f}  in {time.time() - t0:.0f}s")
print(rs.best_params_)

res = pd.DataFrame(rs.cv_results_)
cols = [
    "param_max_depth",
    "param_min_samples_leaf",
    "mean_test_score",
    "std_test_score",
    "rank_test_score",
]
print(res[cols].sort_values("rank_test_score").head(5).to_string(index=False))

**Insight (Part 3):**

**Validation curve.** Underfitting dominates at small C: CV AUC is only about 0.829 at C = 10⁻⁴ and climbs steeply until about C = 10⁻², then rises more slowly. It levels off into a plateau from about C = 1 to C = 100, where CV AUC stays at about 0.846. The best C by CV is 10.0, but values across the plateau score almost the same, and the differences are far smaller than the CV standard deviation (about 0.013), so the exact choice within it does not matter. Train AUC (about 0.850) stays only about 0.004 above CV AUC, so there is no sign of overfitting even at large C.

**Grid vs random search (Random Forest, 5-fold CV, ROC AUC).**

| | Best CV AUC | Fits | Time |
|---|---|---|---|
| Grid search | 0.8468 | 120 | 179s |
| Random search | 0.8464 | 120 | 133s |

The AUCs differ by 0.0004, much less than the CV std (about 0.011), so the two searches are equivalent on accuracy. Both used the same number of fits, and random search happened to run faster, but that depends on which settings were sampled, so I would not read much into it. Both scores are the best of many noisy results (the winner's curse), so they are optimistic and not final performance.

**Six hyperparameters.** I would use random search. A grid grows multiplicatively: even 4 values per parameter gives 4⁶ = 4,096 combinations, or 20,480 fits with 5-fold CV. Random search keeps the budget fixed (for example 24 or 50 settings) however many parameters there are, and it tries a new value of every parameter on each draw. Since the results above show a flat surface where many settings score alike, a modest random sample is enough to find a good region.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

spw = (y_tr == 0).sum() / (y_tr == 1).sum()
print(f'scale_pos_weight = {spw:.2f}')

xgb = XGBClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    early_stopping_rounds=100,
    random_state=42,
    n_jobs=-1
)

xgb.fit(
    X_tr, y_tr,
    eval_set=[(X_tr, y_tr), (X_val, y_val)],
    verbose=False
)

print('Best number of trees:', xgb.best_iteration + 1)

val_auc = roc_auc_score(
    y_val,
    xgb.predict_proba(X_val)[:, 1]
)

print(f'Validation AUC: {val_auc:.4f}')

In [ ]:
ev = xgb.evals_result()

plt.plot(ev['validation_0']['logloss'], label='Train log-loss')
plt.plot(ev['validation_1']['logloss'], label='Validation log-loss')
plt.axvline(xgb.best_iteration, color='k', ls='--', label='Early stop')

plt.xlabel('Boosting round')
plt.ylabel('Log-loss')
plt.legend()
plt.title('XGBoost: when to stop adding trees')
plt.show()

In [ ]:
xdist = {
    'max_depth': randint(2, 7),
    'learning_rate': loguniform(0.01, 0.3),
    'n_estimators': randint(100, 600),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.5, 0.5),
    'min_child_weight': randint(1, 10),
    'reg_lambda': loguniform(0.1, 10)
}

xs = RandomizedSearchCV(
    XGBClassifier(
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    ),
    xdist,
    n_iter=30,
    cv=cv,
    scoring='roc_auc',
    random_state=42
)

xs.fit(X_train, y_train)

print(f'XGBoost tuned CV AUC: {xs.best_score_:.4f}')

print({
    k: round(v, 3) if isinstance(v, float) else v
    for k, v in xs.best_params_.items()
})

In [ ]:
imp = pd.Series(
    xs.best_estimator_.feature_importances_,
    index=X.columns
)

imp.sort_values().tail(10).plot.barh(
    title='XGBoost feature importance (gain)'
)

plt.show()

### XGBoost Results

- Early stopping selected **216 trees**.
- After this point, the validation log-loss stops improving and begins to fluctuate/increase, so additional trees do not improve validation performance.
- Tuned XGBoost CV AUC: **0.8504**.
- Logistic Regression CV AUC: **[insert Part 2 value]**
- Random Forest CV AUC: **[insert Part 2 value]**
- XGBoost improved over the baseline models by **[calculate after checking the CV standard deviations]**.
- The `scale_pos_weight` in Task 4.1 was **2.77**, which gives more weight to the minority churn class and can improve recall for churn customers.

In [ ]:
seg_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

svc = [
    'PhoneService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies'
]

seg = df[seg_cols].copy()

seg['n_services'] = (df[svc] == 'Yes').sum(axis=1)

Z = StandardScaler().fit_transform(seg)   # distances need equal scales

print(seg.describe().round(1))

In [ ]:
ks = range(2, 9)

inertia, sil = [], []

for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Z)
    inertia.append(km.inertia_)
    sil.append(silhouette_score(Z, km.labels_, sample_size=3000,
                                random_state=42))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].plot(ks, inertia, 'o-')
ax[0].set_title('Elbow: inertia (WCSS)')

ax[1].plot(ks, sil, 'o-')
ax[1].set_title('Silhouette score')

for a in ax:
    a.set_xlabel('k')

plt.show()

In [ ]:
k = 4   # choose from the elbow and silhouette plots, then justify it

km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Z)

seg['cluster'] = km.labels_

# Churn was NOT used to form clusters. We use it only to profile them.
profile = seg.assign(churn=y).groupby('cluster').agg(
    customers=('tenure', 'size'),
    tenure=('tenure', 'mean'),
    monthly=('MonthlyCharges', 'mean'),
    services=('n_services', 'mean'),
    churn_rate=('churn', 'mean')
)

print(profile.round(2).sort_values('churn_rate', ascending=False))

### Choosing k = 4

The elbow plot shows a large reduction in inertia from k=2 to k=3, followed by progressively smaller reductions as k increases, indicating an elbow around k=3–4. The silhouette score is highest at k=2, but k=4 still provides a reasonably strong silhouette score while allowing the customer base to be divided into more meaningful and distinct segments. From a business perspective, using four clusters provides useful differentiation between newer customers, higher-spending customers, and longer-tenure customers with different service levels. Therefore, **k=4** was selected as a practical balance between cluster separation and useful customer segmentation.

### Customer Segment Profiles

| Segment                       |  Size | Avg. tenure | Avg. monthly charge | Avg. services | Churn rate | Retention action                                                                 |
| ----------------------------- | ----: | ----------: | ------------------: | ------------: | ---------: | -------------------------------------------------------------------------------- |
| **Mid-tenure, high-spend**    | 2,157 |       18.38 |               80.41 |          3.28 |        43% | Offer a personalized retention discount or service bundle.                       |
| **New, low-service**          | 1,918 |        8.96 |               37.71 |          1.20 |        32% | Provide an early-tenure onboarding check-in and introductory service offer.      |
| **Long-tenure, high-service** | 1,938 |       59.83 |               92.09 |          5.06 |        14% | Offer loyalty benefits or a multi-service loyalty package.                       |
| **Long-tenure, low-spend**    | 1,030 |       53.61 |               30.96 |          1.48 |         5% | Maintain loyalty engagement with low-cost loyalty rewards and service check-ins. |

The churn rates above are used only to **profile the clusters after K-Means was fitted**; churn was not used as an input to the clustering model.


In [ ]:
Xs = StandardScaler().fit_transform(X_train)

pca = PCA().fit(Xs)

cum = np.cumsum(pca.explained_variance_ratio_)

print('Components for 90% variance:', np.argmax(cum >= 0.90) + 1,
      'of', X_train.shape[1])

plt.bar(range(1, 16), pca.explained_variance_ratio_[:15],
        label='Per component')

plt.plot(range(1, 16), cum[:15], 'o-', color='C1',
         label='Cumulative')

plt.xlabel('Principal component')
plt.ylabel('Explained variance ratio')
plt.legend()
plt.title('Scree plot')
plt.show()

### Interpretation
15 principal components are required to retain at least 90% of the variance, out of the original 30 features. The first few components explain the largest share of the variance, while the contribution of later components decreases.

In [ ]:
P = PCA(n_components=2).fit(Xs)
T = P.transform(Xs)

plt.figure(figsize=(7, 5))
plt.scatter(T[:, 0], T[:, 1], c=y_train, cmap='coolwarm',
            s=6, alpha=0.5)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Customers in 2D, colored by churn (red = churned)')
plt.show()

load = pd.Series(P.components_[0], index=X.columns)

print('PC1 top loadings:')
order = load.abs().sort_values(ascending=False).index
print(load.reindex(order).head(6).round(3))

### PCA Interpretation

15 of the 30 principal components are needed to explain at least 90% of the variance. This suggests that the dataset contains some redundancy, since the information in 30 original features can be represented by 15 components while retaining most of the variance.

PC1 is mainly associated with the `InternetService_No` feature and several `"No internet service"` dummy variables, which have identical top loadings of 0.302. This indicates that these features contain redundant information.

In the 2D PCA scatter plot, churners and non-churners overlap substantially, although there is some separation along the principal components. This suggests that churn is not easily separated using these features alone and that customer churn prediction is a relatively complex problem.


In [ ]:
candidates = {
    'LR (tuned C)': lr.set_params(model__C=Cs[va.mean(axis=1).argmax()]),
    'RF (random search)': rs.best_estimator_,
    'XGBoost (tuned)': xs.best_estimator_,
}

rows = []
for name, m in candidates.items():
    cvs = cross_validate(m, X_train, y_train, cv=cv, scoring='roc_auc')
    rows.append({
        'model': name,
        'cv_auc': cvs['test_score'].mean(),
        'cv_std': cvs['test_score'].std()
    })

cmp = pd.DataFrame(rows)
print(cmp.round(4).to_string(index=False))

### Interpretation

XGBoost achieved the highest mean cross-validated AUC of **0.8504**, followed by Logistic Regression and Random Forest, both with **0.8464**. The CV standard deviations are relatively small, indicating consistent performance across the folds. Therefore, XGBoost will be carried forward as the final candidate for testing.


In [ ]:
best_name = cmp.loc[cmp['cv_auc'].idxmax(), 'model']  # chosen by CV only
best = candidates[best_name].fit(X_train, y_train)

prob = best.predict_proba(X_test)[:, 1]
pred = (prob >= 0.5).astype(int)

print(f'{best_name} on the test set (used once):')
print(f'AUC {roc_auc_score(y_test, prob):.4f}  '
      f'recall {recall_score(y_test, pred):.3f}  '
      f'precision {precision_score(y_test, pred):.3f}')

joblib.dump(best, 'churn_model.joblib')

### Final Model Selection and Evaluation

**XGBoost (tuned)** was chosen because it had the highest cross-validated AUC of **0.8504**, compared with **0.8464** for both Logistic Regression and Random Forest.

On the held-out test set, XGBoost achieved an AUC of **0.8478**. The CV mean ± 2 standard deviations is:

**0.8504 ± 2(0.0125) = 0.8254 to 0.8754**

The test AUC of **0.8478** falls within this range, indicating that the test performance is consistent with the cross-validation results.

In Week 2, the best model had an AUC of approximately **0.842**. The tuned XGBoost model achieved **0.8478**, an improvement of approximately **0.0058 AUC points**. Therefore, the tuning produced a small improvement rather than a large performance gain.


### Final Questions

**1. Is Model A better?**
Not necessarily. The standard error of accuracy is approximately `SE = √[p(1−p)/n]`; for A, `SE ≈ √[0.812(0.188)/1409] ≈ 0.0104`, and for B, `SE ≈ 0.0101`. The accuracy difference is only `0.812 − 0.803 = 0.009`, which is smaller than the uncertainty from the test sample, so the scores are not clearly different based on this information alone.

**2. Why must StandardScaler be inside the Pipeline when you cross-validate?**
It must be inside the Pipeline so the scaler is fitted only on each training fold and then applied to that fold's validation data. This prevents data leakage, which could make cross-validation performance look better than it really is.

**3. How many model fits?**
There are `5 × 4 × 3 = 60` hyperparameter combinations. With 5-fold CV, the total number of model fits is `60 × 5 = 300`.

**4. Why is `gs.best_score_` an optimistic estimate of real performance?**
`gs.best_score_` is the best CV score among many tested hyperparameter combinations, so selecting the maximum introduces some selection bias. It can therefore be slightly higher than the model's performance on truly unseen data.

**5. What does each new tree try to predict in gradient boosting?**
Each new tree tries to predict the current model's errors (more precisely, the negative gradient of the loss function). This is analogous to gradient descent because each tree moves the model in a direction that reduces the loss.

**6. Why would K-means on unscaled features cluster mostly on TotalCharges?**
K-means uses distances, so a feature with a much larger numerical scale can dominate the distance calculation. Since `TotalCharges` has much larger values than many other features, it could disproportionately determine the clusters unless the features are scaled.


In [ ]:
import joblib
best_name = cmp.loc[cmp['cv_auc'].idxmax(), 'model']
best = candidates[best_name].fit(X_train, y_train)
joblib.dump(best, '/kaggle/working/churn_model.joblib')
print("done")

## Task 1.1: What exactly did we save?

In [ ]:
import json
import joblib
import sklearn
import numpy as np
import pandas as pd

model = joblib.load('/kaggle/working/churn_model.joblib')

cols = list(model.feature_names_in_)   # the exact training columns, in order

final = model[-1] if hasattr(model, 'steps') else model

print('Model          :', type(final).__name__)
print('Feature columns:', len(cols))
print('scikit-learn   :', sklearn.__version__)   # you will pin this version

### Task 1.1 Result

The saved model is an **XGBClassifier** with **30 feature columns**.  
The scikit-learn version used is **1.6.1**, which will be pinned in Part 2.

## Task 1.2: The Serving-Time Encoder

In [ ]:
RAW_COLS = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
            'PhoneService', 'MultipleLines', 'InternetService',
            'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
            'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
            'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges',
            'TotalCharges']

def prepare_input(raw, columns):
    """Turn raw customer rows into exactly the columns the model saw."""
    X = pd.get_dummies(raw[RAW_COLS])
    return X.reindex(columns=columns, fill_value=0).astype(float)

### Task 1.2 Result

The `prepare_input` function converts raw customer data into the same feature columns expected by the saved model. It uses one-hot encoding without `drop_first` and reindexes the result to exactly match the model's training columns, filling any missing columns with 0.

### Task 1.3: Reproduce the bug

Score one two-year-contract customer the way Week 3 built features, then the correct way (slide 11).

In [ ]:
print(type(model))
print(len(cols), cols[:5])
print(df['Contract'].unique())

In [ ]:
one = df[df['Contract'] == 'Two year'].head(1)

wrong = build_features(one).reindex(columns=cols, fill_value=0)  # Week 3 way
right = prepare_input(one, cols)

print('Contract columns, wrong:', wrong.filter(like='Contract').values)
print('Contract columns, right:', right.filter(like='Contract').values)
print(f'P(churn) wrong: {model.predict_proba(wrong)[0, 1]:.3f}')
print(f'P(churn) right: {model.predict_proba(right)[0, 1]:.3f}')

In [ ]:
def prepare_input(X, columns):
    X = build_features(X, drop_first=False)   # keep ALL categories, drop none
    return X.reindex(columns=columns, fill_value=0).astype(float)

**Why the fix works:** `prepare_input` encodes with the category list fixed from training, so the encoding no longer depends on which rows are in the batch. A single row now gets the same columns as the full training set, and `Contract_Two year` is correctly set to 1.

### Task 1.4: The parity test

This must pass with a largest difference of 0 (or around 1e-15). If it fails, do not continue: your app would give different answers from your notebook.

In [ ]:
raw = df.drop(columns=['Churn'])

served = prepare_input(raw, cols)

p_train = model.predict_proba(X[cols])[:, 1]   # how the model was evaluated
p_serve = model.predict_proba(served)[:, 1]    # how the app will call it

print('Largest difference:', np.abs(p_train - p_serve).max())

assert np.allclose(p_train, p_serve), 'Training-serving skew detected!'

print(f'Parity test passed on {len(raw)} customers')

### Task 1.4 Result

The parity test passed successfully. The largest difference between the training-time and serving-time predictions is effectively zero, confirming that there is no training-serving skew.

## Task 1.5: What-If Analysis

In [ ]:
def what_if(model, row, columns, changes):
    base = model.predict_proba(prepare_input(row, columns))[0, 1]
    rows = []
    for feature, value in changes:
        alt = row.copy()
        alt[feature] = value
        p = model.predict_proba(prepare_input(alt, columns))[0, 1]
        rows.append({
            'change': f'{feature} = {value}',
            'p_churn': round(p, 3),
            'delta': round(p - base, 3)
        })
    return base, pd.DataFrame(rows)

row = raw.iloc[[int(np.argmax(p_serve))]]      # the riskiest customer

base, table = what_if(model, row, cols, [
    ('Contract', 'One year'),
    ('Contract', 'Two year'),
    ('PaymentMethod', 'Credit card (automatic)')
])

print(f'Baseline P(churn) = {base:.3f}')
print(table.to_string(index=False))

### Task 1.5 Result

The riskiest customer's baseline probability of churn is **0.942 (94.2%)**.

The what-if analysis shows that changing the contract to a **One year** contract reduces the predicted churn probability to **0.875**, a decrease of **0.066**. Changing it to a **Two year** contract reduces the probability further to **0.753**, a decrease of **0.189**. Changing the payment method to **Credit card (automatic)** reduces the probability to **0.923**, a smaller decrease of **0.019**.

Among the tested changes, switching to a **Two year contract** has the largest effect on reducing the predicted churn probability.

## Task 1.6: Save Metadata and a Sample File

In [ ]:
import json

meta = {
    'model_name': type(final).__name__,
    'sklearn_version': sklearn.__version__,
    'feature_columns': cols,
    'threshold': 0.30,
    'cv_auc': 0.8504,
    'cv_auc_std': 0.0125,
    'test_auc': 0.8478,
    'training_data': 'IBM Telco Customer Churn, 7,043 customers',
    'version': '1.0',
}

with open('/kaggle/working/model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved model_meta.json with', len(cols), 'columns')

sample = df.drop(columns=['Churn']).sample(50, random_state=1)
sample.to_csv('/kaggle/working/sample_customers.csv', index=False)

print('Saved 50 customers for the batch demo')

### Task 1.6 Result

The model metadata was saved successfully with **30 feature columns**. The **actual Week 3 results** were used for the CV AUC (**0.8504**), CV standard deviation (**0.0125**), and test AUC (**0.8478**), instead of the example values provided in the task. A sample file containing **50 customers** was also saved successfully for the batch demo.

The two probabilities from Task 1.3 were **0.133** and **0.012**. They differ because the first input incorrectly represents the customer's contract information, while the second correctly matches the one-hot encoded feature representation expected by the model. The parity test passed, with the largest difference between training-time and serving-time predictions being **0.0**, confirming that there was no training-serving skew.

From the what-if analysis, changing the customer's contract to a **Two year** contract lowered the predicted churn probability the most, from **0.942 to 0.753**, a decrease of **0.189**. However, this does not mean the customer would actually stay because the model only predicts the likelihood of churn based on the input features; changing one feature does not guarantee a change in the customer's real-world behavior.